In [1]:
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from scipy.stats import norm

# =========================================================
# WEEK 10 — FUNCTION 7 (STRICT [0,1] BOUNDS, TRANSPARENT BO)
# - Interpretable surrogate: Gaussian Process (GP)
# - Global candidates: Sobol (space-filling)
# - Local candidates: Gaussian around best (trust-region flavor)
# - Min-distance filter: avoids duplicates / wasted queries
# - Mix EI (exploit) + UCB (explore) with logged components
# - Prints x_next with 6 decimals
# =========================================================

# -----------------------------
# 1) Data
# -----------------------------
X_raw = np.array([
    [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
    [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
    [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
    [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
    [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
    [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
    [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
    [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
    [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
    [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
    [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
    [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
    [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
    [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
    [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
    [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
    [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
    [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
    [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
    [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
    [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
    [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
    [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
    [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
    [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
    [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
    [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
    [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
    [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
    [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
    [1.04245 , 1.024693, 1.02457 , 1.061017, 1.098654, 1.051013],
    [0.019976, 0.432955, 0.301662, 0.169496, 0.348651, 0.743371],
    [0.611853, 0.139495, 0.292145, 0.366362, 0.45607 , 0.785175],
    [0.015006, 0.390905, 0.178469, 0.119929, 0.088415, 0.904408],
    [0.046821, 0.309546, 0.608802, 0.064364, 0.39334 , 0.990644],
    [0.011478, 0.62027 , 0.525606, 0.053535, 0.52488 , 0.666127],
    [0.028679, 0.235471, 0.148723, 0.076614, 0.11285 , 0.837107],
    [0.069198, 0.39455 , 0.352452, 0.093928, 0.370707, 0.725655],
    [0.000000, 0.367783, 0.346341, 0.052998, 0.363011, 0.730958]
])

y_raw = np.array([
    6.04432696e-01, 5.62753067e-01, 7.50323668e-03, 6.14243025e-02,
    2.73046801e-01, 8.37465723e-02, 1.36496830e+00, 9.26449549e-02,
    1.78695987e-02, 3.35649360e-02, 7.35163042e-02, 2.06309698e-01,
    8.82563400e-03, 2.68400317e-01, 6.11525528e-01, 1.47981826e-02,
    2.74892508e-01, 6.67632469e-02, 4.21183545e-02, 2.70146502e-03,
    1.82090730e-02, 7.01602756e-03, 1.00506611e-01, 4.75395516e-01,
    6.75141631e-01, 5.16457219e-01, 3.77747962e-03, 3.13433331e-03,
    2.13425228e-02, 9.54111589e-02, 4.636858051500375e-06, 1.680828424430851,
    1.1170576710554418, 0.44099891630237703, 0.8950628737420184, 0.664856997347448,
    0.6583383225997628, 1.6173276124769211, 1.3305832886811908
])

# -----------------------------
# 2) Config (Week 10)
# -----------------------------
RANDOM_STATE = 123
np.random.seed(RANDOM_STATE)

# candidate search
N_GLOBAL = 6000          # smaller but higher quality via Sobol
N_LOCAL  = 2000
LOCAL_SCALE = 0.06       # tighter trust region around best
MIN_DIST = 5e-4          # avoid duplicates / near-duplicates

# acquisition knobs
EI_XI = 0.01
UCB_BETA = 1.8
MIX_ALPHA = 0.70         # slightly more exploit now that we have a strong incumbent

# -----------------------------
# 3) Acquisition helpers
# -----------------------------
def gaussian_ei(mu, sigma, f_best, xi=0.0):
    sigma = np.maximum(sigma, 1e-12)
    z = (mu - f_best - xi) / sigma
    return (mu - f_best - xi) * norm.cdf(z) + sigma * norm.pdf(z)

def gaussian_ucb(mu, sigma, beta=1.0):
    return mu + beta * np.maximum(sigma, 1e-12)

def zscore(a):
    s = np.std(a)
    return (a - np.mean(a)) / s if s > 1e-12 else (a - np.mean(a))

# -----------------------------
# 4) Candidate generation (transparent + reproducible)
# -----------------------------
def sobol_global_candidates(n, d, seed):
    eng = torch.quasirandom.SobolEngine(dimension=d, scramble=True, seed=seed)
    return eng.draw(n).numpy()

def local_candidates(x_best, n, scale, seed):
    rng = np.random.RandomState(seed)
    Xl = x_best + rng.normal(0.0, scale, size=(n, x_best.size))
    return np.clip(Xl, 0.0, 1.0)

def min_dist_filter(Xcand, Xtrain, thr):
    thr2 = thr * thr
    dist2 = ((Xcand[:, None, :] - Xtrain[None, :, :]) ** 2).sum(axis=2)
    keep = dist2.min(axis=1) > thr2
    return Xcand[keep]

# -----------------------------
# 5) Main
# -----------------------------
def main():
    # strict training bounds (handles your >1 row cleanly)
    X = np.clip(X_raw, 0.0, 1.0)
    d = X.shape[1]

    # best observed
    best_idx = int(np.argmax(y_raw))
    x_best = X[best_idx].copy()
    f_best = float(np.max(y_raw))

    # scale inputs/outputs for stable GP fitting
    xs = StandardScaler()
    ys = StandardScaler()
    Xs = xs.fit_transform(X)
    ys_scaled = ys.fit_transform(y_raw.reshape(-1, 1)).ravel()

    # Interpretable surrogate: GP (lengthscales ~ feature sensitivity)
    kernel = (
        C(1.0, (1e-2, 1e2))
        * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5)
        + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-8, 1e-1))
    )
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=False,
        random_state=RANDOM_STATE,
        n_restarts_optimizer=0
    )
    gp.fit(Xs, ys_scaled)

    # candidates = global (Sobol) + local (trust region)
    Xg = sobol_global_candidates(N_GLOBAL, d, RANDOM_STATE)
    Xl = local_candidates(x_best, N_LOCAL, LOCAL_SCALE, RANDOM_STATE)
    Xcand = np.vstack([Xg, Xl])
    Xcand = min_dist_filter(Xcand, X, MIN_DIST)

    # predict mean/std in original y space
    mu_s, std_s = gp.predict(xs.transform(Xcand), return_std=True)
    mu = ys.inverse_transform(mu_s.reshape(-1, 1)).ravel()
    sigma = np.maximum(std_s * float(ys.scale_[0]), 1e-12)

    # acquisition
    ei = gaussian_ei(mu, sigma, f_best, xi=EI_XI)
    ucb = gaussian_ucb(mu, sigma, beta=UCB_BETA)
    score = MIX_ALPHA * zscore(ei) + (1.0 - MIX_ALPHA) * zscore(ucb)

    best_cand = int(np.argmax(score))
    x_next = np.round(np.clip(Xcand[best_cand], 0.0, 1.0), 6)

    # transparent logs
    print("CURRENT BEST OBSERVED")
    print("f_best =", f_best)
    print("x_best =", np.round(x_best, 6))
    print("\nSURROGATE (GP) KERNEL")
    print(gp.kernel_)
    if hasattr(gp.kernel_, "k1") and hasattr(gp.kernel_.k1, "k2"):
        # k1 = Constant * Matern; matern lengthscales live in k1.k2.length_scale
        ls = np.array(gp.kernel_.k1.k2.length_scale, dtype=float).ravel()
        print("lengthscales (smaller => more sensitive):", np.round(ls, 4))

    print("\nACQUISITION BREAKDOWN @ x_next")
    print("mu =", float(mu[best_cand]))
    print("sigma =", float(sigma[best_cand]))
    print("EI =", float(ei[best_cand]))
    print("UCB =", float(ucb[best_cand]))

    np.set_printoptions(suppress=True, formatter={"float_kind": lambda v: f"{v:.6f}"})
    print("\nRECOMMENDED NEXT POINT")
    print("x_next =", x_next)

if __name__ == "__main__":
    main()


CURRENT BEST OBSERVED
f_best = 1.680828424430851
x_best = [0.019976 0.432955 0.301662 0.169496 0.348651 0.743371]

SURROGATE (GP) KERNEL
0.758**2 * Matern(length_scale=[4.25, 100, 100, 1.26, 0.721, 1.11], nu=2.5) + WhiteKernel(noise_level=0.0335)
lengthscales (smaller => more sensitive): [  4.2502 100.     100.       1.2617   0.7206   1.1064]

ACQUISITION BREAKDOWN @ x_next
mu = 1.5655154539790885
sigma = 0.11864790940793463
EI = 0.00887220971442617
UCB = 1.7790816909133709

RECOMMENDED NEXT POINT
x_next = [0.000000 0.319509 0.283024 0.207785 0.339392 0.737674]


C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
